# Does dense retrieval find matches that TF-IDF misses?

A 20-minute decision, run BEFORE committing 8 hours to a full dense build.

Measured on **train**, where ground truth exists -- the test set has no
labels, so recall cannot be measured there at all.

1. TF-IDF pair recall at 45 candidates/S1, via the shipped `SparseBlocker`
   at the same shard settings, so the baseline is the real one
2. dense pair recall, `intfloat/multilingual-e5-small`
3. **union** recall -- the number that matters
4. **the precision of candidates dense ADDS that TF-IDF missed**

Item 4 is the one that decides it. Dense can lift union recall while
adding mostly wrong candidates, which would cost precision and score
worse while looking like a win on recall alone.

Multilingual because India is 810k of the 1.73M test S1 and
Devanagari/Latin transliteration is exactly what a word-level TF-IDF space
cannot see.

**Kill criterion, also printed at the end:** union gain below +1.5 recall
points, or new-candidate precision below 25%, means NO-GO and submission A
stands. A NO-GO answer costs 20 minutes; a wrong GO costs 8 hours.


In [ ]:
!pip -q install sentence-transformers torch numpy scikit-learn scipy polars gdown
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU, then rerun.')
print(torch.cuda.get_device_name(0))

## Unpack the code under test

Byte-identical to the tree that produced submission A.

In [ ]:
import base64, tarfile, io, os
B64 = (
    "..."
)
D = '/content/AmazonML'
os.makedirs(D, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz').extractall(D, filter='data')
os.chdir(D)
import sys; sys.path.insert(0, D)
print('unpacked ->', D, sorted(os.listdir(D)))


## Train split

In [ ]:
import gdown, os
TR = '/content/AmazonML/dataset/train'
os.makedirs(TR, exist_ok=True)
!gdown --folder "https://drive.google.com/drive/folders/1BfqxOy-J6gzgUtiOurfx7GiOo9oe5hsg" -O /content/AmazonML/dataset/train 2>&1 | tail -2
!ls -la /content/AmazonML/dataset/train/
for f in ('train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv',
          'train_ground_truth.tsv'):
    assert os.path.exists(TR + '/' + f), 'missing ' + f
print('train split verified')


In [ ]:
# ---- current blocker: sparse TF-IDF, same config as the shipped run ----
import os, sys, base64, io, tarfile
sys.path.insert(0, ROOT)
PY = ROOT + '/code/business_entity_resolution/src'
if not os.path.isdir(PY):
    raise SystemExit('code not unpacked - run the unpack cell first')
os.environ['ER_POOL_SHARD'] = '1000000'
os.environ['ER_POOL_SHARD_CHAR'] = '400000'
from code.business_entity_resolution.src.blocking_sparse import SparseBlocker, blocking_recall

import time
t0 = time.time()
tfidf_cands = SparseBlocker(verbose=True).candidates_for_country(
    queries, records, max_candidates=45)
print(f'TF-IDF blocking: {time.time()-t0:.0f}s')
rc = blocking_recall(tfidf_cands, truth)
print(f"TF-IDF   pair recall {rc['pair_recall']*100:.2f}%  "
      f"entity full coverage {rc['entity_full_coverage']*100:.2f}%")
TFIDF_RECALL = rc['pair_recall']

In [ ]:
# ---- dense channel: multilingual-e5-small ----
import numpy as np, time
from sentence_transformers import SentenceTransformer

MODEL = 'intfloat/multilingual-e5-small'
m = SentenceTransformer(MODEL, device='cuda')
m.max_seq_length = 64
print('loaded', MODEL, '| dim', m.get_sentence_embedding_dimension())

def embed(items, batch=512):
    # e5 wants asymmetric prefixes: queries vs passages
    return m.encode([t for _, t in items], batch_size=batch,
                    normalize_embeddings=True, show_progress_bar=False,
                    convert_to_numpy=True).astype('float32')

t0 = time.time()
Q = embed([(k, 'query: ' + txt(*v)) for k, v in q_list])
P = embed([(k, 'passage: ' + txt(*v)) for k, v in p_list], batch=256)
print(f'embedded {Q.shape} queries + {P.shape} pool in {time.time()-t0:.0f}s')

K = 45
t0 = time.time()
sims = Q @ P.T
top = np.argpartition(-sims, K, axis=1)[:, :K]
order = np.argsort(-np.take_along_axis(sims, top, axis=1), axis=1)
top = np.take_along_axis(top, order, axis=1)
print(f'retrieved in {time.time()-t0:.0f}s')

pool_ids = [k for k, _ in p_list]
dense_cands = {}
for i, (qid, _) in enumerate(q_list):
    dense_cands[qid] = [(pool_ids[j], float(sims[i, j])) for j in top[i]]
rc = blocking_recall(dense_cands, truth)
print(f"DENSE    pair recall {rc['pair_recall']*100:.2f}%  "
      f"entity full coverage {rc['entity_full_coverage']*100:.2f}%")
DENSE_RECALL = rc['pair_recall']

In [ ]:
# ---- the two numbers that decide everything ----
union = {qid: {c for c, _ in tfidf_cands.get(qid, [])} |
               {c for c, _ in dense_cands.get(qid, [])}
           for qid in queries}
tot = cap = 0
for qid, t in truth.items():
    tot += len(t); cap += len(t & union[qid])
U = cap / tot

# precision of the channel's NEW candidates: of the pairs dense adds that
# TF-IDF missed, what fraction are true matches?
new_tot = new_true = 0
for qid, t in truth.items():
    old = {c for c, _ in tfidf_cands.get(qid, [])}
    for c, _ in dense_cands.get(qid, []):
        if c not in old:
            new_tot += 1; new_true += (c in t)

new_prec = new_true / new_tot if new_tot else 0.0

print('=' * 62)
print(f'  TF-IDF alone     recall {TFIDF_RECALL*100:6.2f}%')
print(f'  dense alone      recall {DENSE_RECALL*100:6.2f}%')
print(f'  UNION            recall {U*100:6.2f}%   '
      f'(+{(U-TFIDF_RECALL)*100:.2f} over TF-IDF)')
print(f'  dense NEW cands  precision {new_prec*100:6.2f}%  '
      f'      ({new_true:,} true of {new_tot:,} added)')
print('=' * 62)
gain = (U - TFIDF_RECALL) * 100
verdict = ('GO: build the dense channel for the test set' if (gain >= 1.5 and new_prec >= 0.25)
           else 'NO-GO: dense does not close the gap that matters; keep submission A')
print(verdict)
open('/content/verdict.txt', 'w').write(
    f'{verdict}\ntfidf={TFIDF_RECALL:.4f} dense={DENSE_RECALL:.4f} '
    f'union={U:.4f} new_precision={new_prec:.4f} gain={gain:.2f}\n')
print('\nsaved /content/verdict.txt')